In [2]:
from llama_index.core import SimpleDirectoryReader, Settings
from llama_index.embeddings.text_embeddings_inference import TextEmbeddingsInference
from llama_index.llms.openai_like import OpenAILike
from llama_index.core.node_parser import TokenTextSplitter
import torch
import json
import gc
from pathlib import Path

In [136]:
# Проверка памяти GPU
def check_gpu_memory():
    """Выводит информацию о памяти GPU"""
    if not torch.cuda.is_available():
        print("GPU не доступен")
        return

    allocated = torch.cuda.memory_allocated() / 1024**2
    reserved = torch.cuda.memory_reserved() / 1024**2
    total = torch.cuda.get_device_properties(0).total_memory / 1024**2

    print(f"📊 GPU Memory Status:")
    print(f"   Total: {total:.0f} MB")
    print(f"   Allocated: {allocated:.1f} MB ({allocated/total*100:.1f}%)")
    print(f"   Reserved: {reserved:.1f} MB ({reserved/total*100:.1f}%)")
    print(f"   Free: {total - reserved:.1f} MB")

# check_gpu_memory()

In [137]:
# Очистка памяти перед загрузкой новой модели
def cleanup_memory():
    """Полная очистка памяти GPU"""
    # Удаляем все большие объекты
    for obj in gc.get_objects():
        try:
          if isinstance(obj, torch.Tensor):
              del obj
        except:
          pass

    # Очищаем кэш
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Сборщик мусора
    gc.collect()

    torch.cuda.ipc_collect()

    print("✅ Память очищена")
    check_gpu_memory()

cleanup_memory()

✅ Память очищена
📊 GPU Memory Status:
   Total: 15832 MB
   Allocated: 0.0 MB (0.0%)
   Reserved: 0.0 MB (0.0%)
   Free: 15831.6 MB


/tmp/ipykernel_149027/2524379139.py:7: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  if isinstance(obj, torch.Tensor):


In [138]:
COLLECT_NAME = "wb_oferta_pvz"
EMBED_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"

In [139]:
# подключение embedding модели с CPU
Settings.embed_model = TextEmbeddingsInference(model_name=EMBED_MODEL_NAME,
                                               base_url="http://localhost:8081")

In [3]:
# специальные промпы для Qwen
def completion_to_prompt(completion):
   return f"<|im_start|>system\n<|im_end|>\n<|im_start|>user\n{completion}<|im_end|>\n<|im_start|>assistant\n"

def messages_to_prompt(messages):
    prompt = ""
    for message in messages:
        if message.role == "system":
            prompt += f"<|im_start|>system\n{message.content}<|im_end|>\n"
        elif message.role == "user":
            prompt += f"<|im_start|>user\n{message.content}<|im_end|>\n"
        elif message.role == "assistant":
            prompt += f"<|im_start|>assistant\n{message.content}<|im_end|>\n"

    if not prompt.startswith("<|im_start|>system"):
        prompt = "<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n" + prompt

    prompt = prompt + "<|im_start|>assistant\n"

    return prompt

In [ ]:
# # подключение qwen3-4b-fp8 llm из vLLM
# Settings.llm = OpenAILike(
#     messages_to_prompt=messages_to_prompt,
#     completion_to_prompt=completion_to_prompt,
#     model="qwen3-4b",                    
#     api_base="http://localhost:8000/v1", 
#     # api_key="dummy",                     
#     temperature=0.0,
#     max_tokens=512,                    
#     timeout=120,                        
#     additional_kwargs={
#         "top_p": 0.8,
#         "frequency_penalty": 0.0,
#         "presence_penalty": 0.0,
#     }
# )



In [4]:
# подключение qwen3-14b-awq llm из vLLM
Settings.llm = OpenAILike(
    messages_to_prompt=messages_to_prompt,
    completion_to_prompt=completion_to_prompt,
    model="qwen3-14b",                    
    api_base="http://localhost:8000/v1", 
    # api_key="dummy",                     
    temperature=0.0,
    max_tokens=512,                    
    timeout=120,                        
    additional_kwargs={
        "top_p": 0.8,
        "frequency_penalty": 0.0,
        "presence_penalty": 0.0,
    }
)

/home/yuri/HSE/ВКР/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import torch
import time

def check_cuda():
    print(f"CUDA Available: {torch.cuda.is_available()}")
    if not torch.cuda.is_available():
        return
    
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Allocated: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")
    print(f"Cached: {torch.cuda.memory_reserved(0) / 1024**2:.2f} MB")
    
    try:
        print("Пробуем создать тензор и перекинуть на GPU...")
        x = torch.randn(1000, 1000).cuda()
        print("Успешно! CUDA ядра отвечают.")
        
        print("Тестируем матричное умножение (проверка ядер)...")
        res = torch.matmul(x, x)
        print("Матричное умножение выполнено успешно.")
        
    except Exception as e:
        print(f"ОШИБКА: GPU не отвечает на команды: {e}")

if __name__ == "__main__":
    check_cuda()

CUDA Available: True
Device Name: NVIDIA GeForce RTX 5070 Ti
Allocated: 0.00 MB
Cached: 0.00 MB
Пробуем создать тензор и перекинуть на GPU...
Успешно! CUDA ядра отвечают.
Тестируем матричное умножение (проверка ядер)...
Матричное умножение выполнено успешно.


In [5]:
import requests
try:
    req = requests.get(url="http://localhost:8000/health")
    req.status_code
    print("✅ vLLM успешно подключён к LlamaIndex")
except Exception as e:
    print(f"Ошибка {e}")

Ошибка ('Connection aborted.', ConnectionResetError(104, 'Connection reset by peer'))


In [ ]:
local_data = Path.cwd() / "data"
# загружаем документы постранично
reader = SimpleDirectoryReader(input_dir=local_data, required_exts=[".pdf"])
docs = reader.load_data() # 86 docs = 86 lists of pdf

In [ ]:
# для создания llm_questions
# splitter = TokenTextSplitter(chunk_size=14000, chunk_overlap=2000)
# nodes = splitter.get_nodes_from_documents(docs)

In [145]:
docs[0].text

'ОФЕРТАоб оказании услуг в ПВЗ\n1. Термины и определения\n1.1. Активация ПВЗ — процесс, в результате которого ПВЗ Исполнителя становится действующим, моментом\nАктивации ПВЗ признается получение Исполнителем подтверждения от Заказчика о возможной Активации в\nсоответствии с п. 3.2 Договора.\n1.2. Аудит ПВЗ — действия, совершаемые различными способами Заказчиком по проверке соответствия ПВЗ\nИсполнителя в части соблюдения Регламента состояния ПВЗ.\n1.3. Баланс — сведения, отраженные в Личном кабинете Исполнителя о начисленном Исполнителю\nвознаграждении за оказанные Услуги, с учетом сумм неустоек (штрафов) и возмещаемых Исполнителем\nзадолженностей, а также с учетом суммы иных обязательств Исполнителя перед Заказчиком.\n1.4. Блокировка ПВЗ — частичное ограничение функционала Портала в отношении ПВЗ, применяемое в порядке,\nпредусмотренном Договором. Под частичным ограничением функционала понимаются в том числе временные\nограничения на объем доступных операций и функциональные возможнос

In [ ]:
# import asyncio

# async def generate_questions():
#     promt = """Твоя задача: прочитать текст и составить 2 вопроса (простой и сложный).
#     Твой ответ ДОЛЖЕН быть строго в формате JSON. Не пиши ничего, кроме JSON.

#     Формат ответа:
#     {"simple": "текст простого вопроса", "complex": "текст сложного вопроса"}

#     Текст для анализа:
#     """
    
#     # создаем список асинхронных задач
#     tasks = [Settings.llm.acomplete(prompt=promt + doc.text) for doc in docs]
    
#     responses = await asyncio.gather(*tasks)
#     return [r.text for r in responses]

# responses = await generate_questions()

In [ ]:
import asyncio

async def generate_questions():
    promt = """Твоя задача: прочитать текст и составить 3 вопроса (простой, средний по сложности и сложный).
    Вопросы должны быть конкретными, строго на тему оферты об указании услуг пункта выдачи заказов.
    Твой ответ ДОЛЖЕН быть строго в формате JSON. Не пиши ничего, кроме JSON.

    Формат ответа:
    {"simple": "текст простого вопроса", "simple_complex": "текст среднего вопроса", "complex": "текст сложного вопроса"}

    Текст для анализа:
    """
    
    # создаем список асинхронных задач
    tasks = [Settings.llm.acomplete(prompt=promt + doc.text) for doc in docs]
    
    responses = await asyncio.gather(*tasks)
    return [r.text for r in responses]

responses = await generate_questions()

In [204]:
responses

['{"simple": "Что такое активация ПВЗ?", "complex": "Какие последствия может иметь блокировка выплат ПВЗ для Исполнителя, и в каких случаях она может быть применена в соответствии с условиями Договора?"}',
 '{"simple": "Кто является Заказчиком в соответствии с текстом?", "complex": "Какие нормативные правовые акты регулируют обработку персональных данных в зависимости от страны, в которой находится Исполнитель, и как это влияет на обязанности сторон в контексте предоставления информации о подмене товара в ПВЗ?"}',
 '{"simple": "Что такое Личный кабинет (ЛК) на Портале?", "complex": "Какие последствия наступают для Исполнителя, если его рейтинг ПВЗ снижается ниже минимального значения, установленного в Договоре, и как это влияет на взаимоотношения между Заказчиком и Исполнителем в контексте обязательств по оказанию услуг?"}',
 '{"simple": "Что такое Поручение?", "complex": "Какие факторы учитываются при формировании рейтинга ПВЗ Исполнителя, и как этот рейтинг влияет на качество оказыва

In [206]:
dict_to_json = {}
for resp in responses:
    resp.strip()
    dict_to_json[f"doc_{responses.index(resp)}"] = resp.strip()
    # print(resp)
    # break

In [ ]:
dict_to_json

In [ ]:
with open("docs_questions.jsonl", "a", encoding="utf-8") as f:
    for k, v in dict_to_json.items():
        model_response_dict = json.loads(v)
        line_data = {k: model_response_dict}
        f.write(json.dumps(line_data, ensure_ascii=False) + "\n")